# PIC_SUB_02: pre-specified replication of the admission subphenotypes in PIC

**Run this notebook on your own computer only.** PIC (Paediatric Intensive Care database, PhysioNet v1.1.0) is credentialed
data. It must not be uploaded to any online service, including chat assistants. The notebook writes aggregate results only.

**What it does.** It carries out the replication in the PIC database that the OSF registration osf.io/ygtr2 lists, as fixed in
the addendum deposited on OSF before this analysis. It fixes the PIC cohort with the pre-specified diagnosis codes and checks the
registered threshold of 200 eligible children. It then assigns each child to a subphenotype with the frozen Wuhan model, without
refitting, and computes the pre-specified outcomes and sensitivity analyses. All of them are reported, whatever they show.

**Before running.**
1. Deposit the addendum, this notebook and the model file on the study's OSF project. OSF records the upload date, which
   shows that the definitions and the code were fixed before the analysis.
2. Put three files in one folder: this notebook, `PIC_SUB_02_frozen_wuhan_model.json` (the frozen model, aggregate values
   only) and a text file `PIC_SUB_02_config.txt` with the addendum link and, if the PIC CSV files are elsewhere, their folder:

       osf_addendum = https://osf.io/xxxxx
       pic_dir = D:\PIC\1.1.0

3. Do not edit the notebook, before or after depositing it. The addendum lists the fingerprints (SHA-256) of its code and of
   the model file. The run checks the model file and records both fingerprints.

**How to run.** *Restart Kernel and Run All Cells* (about 5 to 15 minutes). Requirements: Python 3.9 or later, pandas 2.0 or
later, NumPy and SciPy (the `ae_subpheno` environment works).

**Outputs.**
* `output_pic_sub02/`: shareable after a look. Aggregate tables and `summary.md` in a coarse form: counts rounded to the nearest
  5 (1 to 4 shown as "<5"); percentages and estimates computed from counts are left out (they are in the local tables);
  statistics of continuous data (correlations, ratios of geometric means, means, model fit) are shown. Rounding keeps exact small
  counts from being worked out by differencing.
* `output_pic_sub02_LOCAL_ONLY/`: the same tables without suppression, for the manuscript. Keep this folder on this computer.
  Do not upload or send it, and do not share the executed notebook itself.

In [1]:
# Setup, configuration and fixed parameters (pre-specified; not to be changed).
import os, re, sys, json, time, gzip, hashlib, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import norm, beta
from scipy.optimize import linear_sum_assignment

NOTEBOOK_MARKER = 'PIC_SUB_02_REPLICATION_NOTEBOOK'
sys.dont_write_bytecode = True
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)

HERE = Path.cwd()
CONFIG = {}
_cfg = HERE / 'PIC_SUB_02_config.txt'
if _cfg.exists():
    for _line in _cfg.read_text(encoding='utf-8-sig').splitlines():
        if '=' in _line and not _line.strip().startswith('#'):
            _k, _v = _line.split('=', 1)
            CONFIG[_k.strip().lower()] = _v.strip().strip('"\'')
SRC = Path(CONFIG['pic_dir']) if CONFIG.get('pic_dir') else HERE
FIXTURE = (SRC / 'SYNTHETIC_TEST_FIXTURE.txt').exists()
OSF_ADDENDUM = CONFIG.get('osf_addendum', '')
if not FIXTURE and not re.search(r'osf\.io/\w+', OSF_ADDENDUM):
    raise RuntimeError('Deposit the PIC replication addendum on OSF first, then put its link in PIC_SUB_02_config.txt next to this '
                       'notebook (a line "osf_addendum = https://osf.io/..."). The notebook runs on PIC data only after that.')

SEED = 20261019
N_BOOT = 2000                    # bootstrap resamples (children) for every confidence interval that uses the bootstrap
AGE_MIN_DAYS = 28                # neonates excluded, as in Wuhan
AGE_MAX_YEARS = 18
LAB_WINDOW_H = (-24, 24)         # first value from 24 h before to 24 h after the time anchor (Wuhan)
VALIDATOR_WINDOW_H = (-24, 48)   # cytokines and ferritin (Wuhan)
VITAL_WINDOW_H = (0, 24)         # first charted value within 24 h after the time anchor (Wuhan)
MIN_TESTS = 11                   # of the 22 blood tests (Wuhan clustering population)
GATE_N = 200                     # registered threshold (analysis plan, decision P15)
MIN_S1 = 10                      # pre-specified replication criterion: at least 10 children assigned to S1 ...
MIN_R = 0.80                     # ... whose mean profile correlates at least 0.80 with the Wuhan S1 profile
DE_NOVO_STARTS, DE_NOVO_FALLBACK, DE_NOVO_K_MAX, MIN_CLASS_SHARE = 100, 500, 6, 0.05
CHUNK = 500_000
PKG_FILE = HERE / 'PIC_SUB_02_frozen_wuhan_model.json'
PKG_SHA256 = 'd02b69ead00732b4b46b8bf67ff46dc46e5727eb43c735247b1d57badbb408f5'    # content fingerprint of the frozen model file (listed in the addendum)

OUT = HERE / 'output_pic_sub02'
LOCAL = HERE / 'output_pic_sub02_LOCAL_ONLY'
_ours = re.compile(r'^(\d\d[a-z]?_[\w-]+\.csv|summary\.md|run_info\.json|README_LOCAL_ONLY\.txt)$')
for _d in (OUT, LOCAL):
    _d.mkdir(exist_ok=True)
    for _p in _d.iterdir():                     # remove this notebook's own files from an earlier run (nothing else)
        if _p.is_file() and _ours.match(_p.name):
            _p.unlink()
(LOCAL / 'README_LOCAL_ONLY.txt').write_text('Unsuppressed aggregate tables of PIC_SUB_02, for the authors\' manuscript only. Keep this '
                                             'folder on this computer: do not upload it or send it to anyone or any online service.\n',
                                             encoding='utf-8')
T0 = time.time()
if FIXTURE:
    print('#' * 100 + '\nTEST RUN ON A SYNTHETIC FIXTURE: every number below is meaningless and must never be reported.\n' + '#' * 100)
print('python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)


def code_fingerprint():
    """SHA-256 of this notebook's cell sources (cell type and source of every cell, outputs ignored). The running notebook is
    identified by the path the front end reports (Jupyter: JPY_SESSION_NAME; VS Code: __vsc_ipynb_file__); otherwise the only
    notebook in the folder that carries this notebook's marker. With several candidates, all are recorded."""
    cands = []
    for p in sorted(HERE.glob('*.ipynb')):
        try:
            nb = json.loads(p.read_text(encoding='utf-8'))
        except Exception:
            continue
        cells = nb.get('cells', [])
        srcs = [c['source'] if isinstance(c['source'], str) else ''.join(c['source']) for c in cells]
        if any(NOTEBOOK_MARKER in s_ for s_ in srcs):
            parts = [c['cell_type'] + '\n' + s_.replace('\r\n', '\n') for c, s_ in zip(cells, srcs)]
            cands.append((p.name, hashlib.sha256('\n\x1e\n'.join(parts).encode('utf-8')).hexdigest()))
    own = os.environ.get('JPY_SESSION_NAME') or globals().get('__vsc_ipynb_file__')
    if own:
        for nm, h in cands:
            if nm == Path(str(own)).name:
                return nm, h
    if len(cands) == 1:
        return cands[0]
    if not cands:
        return None, 'notebook file not found next to the outputs'
    return 'several notebook files', 'ambiguous (keep one copy in the folder): ' + '; '.join(f'{nm} {h}' for nm, h in cands)


NB_NAME, CODE_SHA256 = code_fingerprint()
print('Code fingerprint:', NB_NAME, CODE_SHA256)

python 3.12.10 | pandas 2.3.3 | numpy 2.4.4
Code fingerprint: PIC_SUB_02_replication.ipynb 6193ed4ce89128519eb66ab93dd5fd1c16044a152e205a8c0217c027d1e42907


In [2]:
# Cells with a local (unsuppressed) and a shareable (suppressed) form, and table writing.
class Txt:
    """A table cell: `local` is the exact value; `share` is what may leave this computer."""
    __slots__ = ('local', 'share')

    def __init__(self, local, share='__same__'):
        self.local = local
        self.share = local if share == '__same__' else share

    def __repr__(self):
        return f'Txt({self.local!r}, {self.share!r})'


def small(n):
    return n is not None and not (isinstance(n, float) and np.isnan(n)) and 0 < int(round(float(n))) < 5


def ok5(*ns):
    """True when every count given is 0 or at least 5."""
    for n in ns:
        if n is None or (isinstance(n, float) and np.isnan(n)):
            return False
        if 0 < int(round(float(n))) < 5:
            return False
    return True


SUP = 'suppressed'


def r5(n):
    """A count in the shareable form: 0; '<5' for 1-4; otherwise rounded to the nearest 5 (so that no difference of shown
    counts gives an exact count)."""
    n = int(round(float(n)))
    if n == 0:
        return 0
    if n < 5:
        return '<5'
    return int(5 * np.floor(n / 5 + 0.5))


def C(n):
    """A count: exact in the local form, rounded (r5) in the shareable form."""
    return Txt(int(n), r5(n))


def pct_txt(k, n, d=1, **_):
    """Percentage k/n, in the local tables only: a percentage of exact counts can narrow a rounded count down to one value, so the
    shareable tables give the rounded counts instead (blank here)."""
    if not n:
        return Txt(None)
    return Txt(round(100 * k / n, d), '')


def sig2(x):
    """A number to 2 significant digits (shareable form of estimates from continuous data)."""
    if x is None or not np.isfinite(x):
        return 'NA'
    if x == 0:
        return '0'
    v = float(f'{x:.2g}')
    d = 1 - int(np.floor(np.log10(abs(v))))
    return f'{v:.{max(d, 0)}f}'


def ci_local(e, lo, hi, d=2, na='not estimable'):
    """An estimate computed from counts (risk, risk ratio, odds ratio, accuracy) with its CI: local tables only (blank here)."""
    if e is None or not np.isfinite(e):
        return Txt(na, '')
    return Txt(fmt_ci(e, lo, hi, d), '')


def ci_est(e, lo, hi, d=2, na='not estimable'):
    """An estimate from continuous data (ratio of geometric means) with its CI: d decimals locally, 2 significant digits in the
    shareable form."""
    if e is None or not np.isfinite(e):
        return Txt(na, na)
    return Txt(fmt_ci(e, lo, hi, d), f'{sig2(e)} ({sig2(lo)}-{sig2(hi)})')


def est(local, share, hide=False):
    """A statistic of continuous data in two forms; suppressed in the shareable form when hidden."""
    return Txt(local, SUP if hide else share)


def num(x, d=2, hide=False):
    """A statistic of continuous data (a mean, a correlation): d decimals in both forms, suppressed when hidden."""
    if x is None or not np.isfinite(x):
        return Txt(None)
    v = round(float(x), d)
    return Txt(v, SUP if hide else v)


def share_print(rows, drop=()):
    """Print a table in its shareable form only, so that the notebook's own outputs hold no exact count."""
    df = render(rows, 'share')
    print(df.drop(columns=[c for c in drop if c in df]).to_string(index=False))


def render(rows, mode):
    """Table in the local or shareable form (object columns, so that counts stay integers)."""
    cols = list(dict.fromkeys(k for r in rows for k in r))
    return pd.DataFrame([{k: (getattr(v, mode) if isinstance(v, Txt) else v) for k, v in r.items()} for r in rows],
                        columns=cols, dtype=object)


def save(rows, name):
    """Write a table twice: shareable (suppressed) to output_pic_sub02, exact to output_pic_sub02_LOCAL_ONLY."""
    for mode, folder in (('share', OUT), ('local', LOCAL)):
        render(rows, mode).to_csv(folder / name, index=False, encoding='utf-8-sig')
    return render(rows, 'local')


def q(s):
    """Median (25th-75th percentile) as text; blank for fewer than 5 values."""
    s = pd.Series(s, dtype=float).dropna()
    if len(s) < 5:
        return None
    f = lambda x: f'{x:.3g}'
    return f'{f(s.median())} ({f(s.quantile(.25))}-{f(s.quantile(.75))})'


def cp_ci(k, n):
    """Exact (Clopper-Pearson) 95% CI of a proportion."""
    if not n:
        return (np.nan, np.nan)
    lo = beta.ppf(0.025, k, n - k + 1) if k > 0 else 0.0
    hi = beta.ppf(0.975, k + 1, n - k) if k < n else 1.0
    return float(lo), float(hi)


def fmt_ci(e, lo, hi, d=2):
    if e is None or not np.isfinite(e):
        return None
    lo_s = f'{lo:.{d}f}' if lo is not None and np.isfinite(lo) else 'NA'
    hi_s = f'{hi:.{d}f}' if hi is not None and np.isfinite(hi) else 'NA'
    return f'{e:.{d}f} ({lo_s}-{hi_s})'

## Frozen Wuhan model
The file holds the six-class model of the reported Wuhan run (class weights, means and variances on the standardized scale),
its preprocessing constants (log scale, winsorizing limits, means and SDs), effective Wuhan reference limits by age band, the
Fleming centiles, the S1 bedside rule and Wuhan reference values for comparison. Applying the PIC preprocessing (fixed limits
by age band and the hs-CRP reporting floor) to the Wuhan children reproduced the reported assignment for 92.9% of them
(kappa 0.91). All 53 S1 children stayed S1, and 2 others joined S1. The file's fingerprint must match
the one listed in the addendum.

In [3]:
if not PKG_FILE.exists():
    raise FileNotFoundError(f'{PKG_FILE.name} not found next to this notebook.')
PKG = json.loads(PKG_FILE.read_text(encoding='utf-8'))
_h = PKG.pop('sha256_of_content')
_c = hashlib.sha256(json.dumps(PKG, sort_keys=True, ensure_ascii=True, separators=(',', ':')).encode()).hexdigest()
if not (_h == _c == PKG_SHA256):
    raise RuntimeError('The frozen model file does not match the version listed in the addendum (fingerprint mismatch).')
CLS = PKG['classes']                                    # S1 ... S6 (display order)
IND = PKG['indicators']                                 # 24 clustering indicators, model order
LAB22 = PKG['lab_tests_22']                             # the 22 blood tests of the 11-test rule
MODEL_LABS = [c for c in IND if c in LAB22]             # 21 tests in the model (white cell count is not)
VITALS3 = ['temp_c', 'hr_z', 'rr_z']
VALIDATORS = ['il6', 'il10', 'tnf_alpha', 'ifn_gamma', 'ferritin']
W = np.array(PKG['weights'], float)
MU = np.array(PKG['means'], float)
VAR = np.array(PKG['variances'], float)
PRE = PKG['preprocessing']
REFLIM = PKG['reference_limits']
BAND_EDGES = [(b['name'], b['from_days'], b['to_days']) for b in PKG['age_bands']]
FLEM = PKG['fleming']
RULE = PKG['bedside_rule']
WREF = PKG['wuhan_reference']
K = len(CLS)
S1, S6 = 0, K - 1
LABEL = dict(PKG['indicator_labels'])
LABEL.update({'wbc': 'White cell count', 'il6': 'IL-6', 'il10': 'IL-10', 'tnf_alpha': 'TNF-alpha', 'ifn_gamma': 'IFN-gamma',
              'ferritin': 'Ferritin', 'heart_rate': 'Heart rate', 'resp_rate': 'Respiratory rate'})
assert MU.shape == (K, len(IND)) and VAR.shape == (K, len(IND)) and abs(W.sum() - 1) < 1e-9
print(f'Frozen model: {K} classes, {len(IND)} indicators, fingerprint {PKG_SHA256[:16]}...')

Frozen model: 6 classes, 24 indicators, fingerprint d02b69ead00732b4...


In [4]:
# Files and columns (as in PIC_SUB_01).
ALIASES = {'D_DIAGNOSES_ICD': 'D_ICD_DIAGNOSES'}
FILES = {}
for p in sorted(SRC.iterdir()):
    m = re.match(r'^(.*?)\.csv(\.gz)?$', p.name, flags=re.I)
    if m:
        t = m.group(1).upper()
        FILES[ALIASES.get(t, t)] = p
REQUIRED = ['PATIENTS', 'ADMISSIONS', 'ICUSTAYS', 'DIAGNOSES_ICD', 'D_ICD_DIAGNOSES', 'D_LABITEMS', 'LABEVENTS', 'D_ITEMS', 'CHARTEVENTS']
missing = [t for t in REQUIRED if t not in FILES]
if missing:
    raise FileNotFoundError(f'Required PIC tables not found in {SRC}: {missing}')


def read(table, usecols=None, chunksize=None):
    """Read a PIC table as text with upper-case column names; usecols in upper case (missing ones are skipped)."""
    p = FILES[table]
    head = pd.read_csv(p, nrows=0)
    cmap = {c: c.strip().upper() for c in head.columns}
    cols = [c for c in head.columns if usecols is None or cmap[c] in usecols]
    kw = dict(usecols=cols, dtype=str, keep_default_na=False, na_values=[''], encoding='utf-8')
    if chunksize:
        return (ch.rename(columns=cmap) for ch in pd.read_csv(p, chunksize=chunksize, **kw))
    return pd.read_csv(p, **kw).rename(columns=cmap)


def pick(cols, *names, contains=None):
    cols = list(cols)
    for n in names:
        if n in cols:
            return n
    if contains:
        for c in cols:
            if all(k in c for k in contains):
                return c
    return None


def to_dt(s):
    return pd.to_datetime(s, errors='coerce')


FILE_ROWS = [{'table': t, 'file': FILES[t].name} for t in sorted(FILES) if t in REQUIRED]
save(FILE_ROWS, '01_files.csv')

PAT = read('PATIENTS', ['SUBJECT_ID', 'GENDER', 'DOB'])
ADM = read('ADMISSIONS', ['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'HOSPITAL_EXPIRE_FLAG'])
ICU = read('ICUSTAYS', ['HADM_ID', 'ICUSTAY_ID', 'FIRST_CAREUNIT', 'INTIME'])
PAT['DOB'] = to_dt(PAT.DOB)
for c in ['ADMITTIME', 'DISCHTIME']:
    ADM[c] = to_dt(ADM[c])
ICU['INTIME'] = to_dt(ICU.INTIME)
ADM = ADM.merge(PAT, on='SUBJECT_ID', how='left')
ADM['AGE_DAYS'] = (ADM.ADMITTIME - ADM.DOB).dt.total_seconds() / 86400
ADM['DIED'] = pd.to_numeric(ADM.HOSPITAL_EXPIRE_FLAG, errors='coerce').eq(1)
_g = ADM.GENDER.astype(str).str.strip().str.upper().str[:1]
ADM['SEX'] = np.where(_g.eq('M'), 'male', np.where(_g.eq('F'), 'female', 'unknown'))
FIRST_ICU = ICU.dropna(subset=['INTIME']).sort_values(['HADM_ID', 'INTIME']).drop_duplicates('HADM_ID').set_index('HADM_ID')
ADM['ICU_IN'] = ADM.HADM_ID.map(FIRST_ICU.INTIME)
ADM['FIRST_UNIT'] = ADM.HADM_ID.map(FIRST_ICU.FIRST_CAREUNIT)
ADM = ADM.drop_duplicates('HADM_ID').set_index('HADM_ID', drop=False)
ADM.index.name = None
print(f'Admissions {len(ADM)}, children {ADM.SUBJECT_ID.nunique()}')

Admissions 13449, children 12881


## Cohort (pre-specified definition)
* **Inclusion codes** (discharge diagnoses, any position): acute encephalopathy codes (list A) or acute encephalitis codes
  (list B), fixed after the outcome-blind review of the PIC diagnosis dictionary (PIC_SUB_01).
* **Index admission**: the child's first admission with an inclusion code.
* **Exclusions**: an exclusion code in the index admission (list C: other causes of encephalopathy, sequelae, neonatal
  conditions, post-operative or tuberculous intracranial infection, abscess, myelitis without encephalitis, chemical meningitis);
  younger than 28 days or 18 years or older at the index admission.
* **Eligible**: at least 11 of the 22 blood tests from 24 h before to 24 h after hospital admission.
* **Registered threshold**: the replication goes ahead only with at least 200 eligible children.

In [5]:
LIST_A = {  # acute encephalopathy: code -> (Chinese title in the PIC dictionary, group)
    'G93.400': ('脑病', 'Encephalopathy codes'), 'G92.x00': ('中毒性脑病', 'Encephalopathy codes'),
    'G93.700': ('赖氏综合征', 'Encephalopathy codes'), 'G93.701': ('赖氏综合症', 'Encephalopathy codes')}
LIST_B = {  # acute encephalitis
    'G04.913': ('脑膜脑炎', 'Meningoencephalitis or encephalitis, unspecified'),
    'G04.914': ('脑膜脑炎', 'Meningoencephalitis or encephalitis, unspecified'),
    'G04.916': ('脑炎', 'Meningoencephalitis or encephalitis, unspecified'),
    'A86.x00': ('病毒性脑炎', 'Viral encephalitis'), 'A86.02': ('病毒性脑炎', 'Viral encephalitis'),
    'A86.01': ('病毒性脑膜脑炎', 'Viral encephalitis'), 'A86.x02': ('病毒性脑膜脑炎', 'Viral encephalitis'),
    'A86.x01': ('病毒性脑脊髓炎', 'Viral encephalitis'), 'A83.001': ('流行性乙型脑炎(日本脑炎)', 'Viral encephalitis'),
    'G04.921': ('脑干脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.910': ('脑干脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.908': ('局限性脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.909': ('局灶性脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.912': ('脑脊髓炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.000': ('播散性脑炎', 'ADEM or disseminated encephalitis'), 'G04.001': ('急性播散性脑脊髓炎', 'ADEM or disseminated encephalitis'),
    'G04.201': ('化脓性脑膜脑炎', 'Bacterial meningoencephalitis'), 'G04.807': ('化脓性脑膜脑炎', 'Bacterial meningoencephalitis')}
LIST_C = {  # exclusion codes: code -> (Chinese title, reason)
    'P91.600': ('新生儿缺氧缺血性脑病', 'neonatal condition'), 'P91.900': ('新生儿脑病', 'neonatal condition'),
    'P21.902': ('新生儿缺血缺氧性脑病', 'neonatal condition'), 'P59.901': ('新生儿高胆红素血症性脑病', 'neonatal condition'),
    'G93.101': ('缺氧性脑病（肺性脑病）', 'hypoxic or anoxic injury'), 'K72.903': ('肝昏迷(肝性脑病)', 'hepatic encephalopathy'),
    'I67.400': ('高血压性脑病', 'hypertensive encephalopathy'), 'I67.401': ('高血压性脑病', 'hypertensive encephalopathy'),
    'E16.107+': ('低血糖昏迷性脑病', 'metabolic encephalopathy'), 'E16.108+': ('低血糖性脑病', 'metabolic encephalopathy'),
    'G93.403': ('代谢性脑病', 'metabolic encephalopathy'), 'E51.200+': ('韦尼克脑病', 'metabolic encephalopathy'),
    'E51.201': ('韦尼克氏脑病(急性出血性脑灰质炎)', 'metabolic encephalopathy'), 'N18.801': ('尿毒症性脑病', 'metabolic encephalopathy'),
    'N18.806+': ('尿毒症性脑病', 'metabolic encephalopathy'), 'A81.200': ('进行性多灶性白质脑病', 'leukoencephalopathy'),
    'A81.201': ('播散性坏死性脑白质病', 'leukoencephalopathy'), 'I67.300': ('进行性血管性脑白质病', 'leukoencephalopathy'),
    'I67.301': ('宾斯旺格病', 'leukoencephalopathy'), 'G93.406': ('脑白质病', 'leukoencephalopathy'),
    'B94.100': ('病毒性脑炎后遗症', 'sequelae'), 'B94.101': ('病毒性脑炎后遗症', 'sequelae'), 'B94.102': ('流行性乙型脑炎后遗症', 'sequelae'),
    'F48.909': ('脑病后遗症', 'sequelae'), 'G09.02': ('脑炎后遗症', 'sequelae'), 'G09.x01': ('感染中毒性脑病后遗症', 'sequelae'),
    'T97.02': ('感染中毒后脑病后遗症', 'sequelae'), 'T97.x02': ('中毒性脑病后遗症', 'sequelae'),
    'G93.402': ('低颅压综合症', 'not an acute encephalopathy'), 'F07.201': ('创伤性脑病', 'not an acute encephalopathy'),
    'G93.407': ('脑病变', 'not an acute encephalopathy'), 'M32.114+': ('狼疮性脑病', 'not an acute encephalopathy'),
    'G06.003': ('手术后颅内感染', 'post-operative intracranial infection'), 'G04.801': ('化脓性脊髓炎', 'myelitis without encephalitis'),
    'G04.903': ('急性脊髓炎', 'myelitis without encephalitis'), 'G04.911': ('脑脊髓神经根神经炎', 'myelitis without encephalitis'),
    'G04.805': ('化学性脑膜炎', 'chemical meningitis'), 'A17.801': ('结核性脑膜脑炎', 'tuberculosis of the nervous system'),
    'A17.803+': ('结核性脑膜脑炎', 'tuberculosis of the nervous system'), 'A17.804': ('结核性脑炎', 'tuberculosis of the nervous system'),
    'A17.804+': ('结核性脑炎', 'tuberculosis of the nervous system')}
EXCL_PREFIX = r'^(A17|G06|P91)'     # tuberculosis of the nervous system, intracranial abscess and granuloma, neonatal cerebral disorders
SENS_A_DROP = {'G04.000', 'G04.001', 'G04.201', 'G04.807'}   # sensitivity analysis S-a: ADEM and bacterial meningoencephalitis
GROUP_ORDER = ['Encephalopathy codes', 'Bacterial meningoencephalitis', 'ADEM or disseminated encephalitis', 'Viral encephalitis',
               'Brainstem or focal encephalitis, encephalomyelitis', 'Meningoencephalitis or encephalitis, unspecified']
INC_GROUP = {c: g for c, (_, g) in {**LIST_A, **LIST_B}.items()}

DXD = read('D_ICD_DIAGNOSES')
C_CODE = pick(DXD.columns, 'ICD10_CODE_CN', contains=['CODE', 'CN']) or pick(DXD.columns, 'ICD10_CODE', contains=['CODE'])
C_TCN = pick(DXD.columns, 'TITLE_CN', contains=['TITLE', 'CN'])
TITLE_CN = dict(zip(DXD[C_CODE].astype(str).str.strip(), DXD[C_TCN].astype(str))) if C_TCN else {}
_nows = lambda s: re.sub(r'\s+', '', str(s))
_bad = [(c, t, TITLE_CN.get(c)) for c, (t, _) in {**LIST_A, **LIST_B, **LIST_C}.items()
        if c in TITLE_CN and _nows(TITLE_CN[c]) != _nows(t)]
_absent_inc = [c for c in {**LIST_A, **LIST_B} if c not in TITLE_CN]
if _bad or _absent_inc:
    raise RuntimeError(f'The diagnosis dictionary differs from PIC v1.1.0 as reviewed: titles {_bad}; inclusion codes not found {_absent_inc}')

DX = read('DIAGNOSES_ICD')
C_DXCODE = pick(DX.columns, 'ICD10_CODE_CN', contains=['CODE', 'CN']) or pick(DX.columns, 'ICD10_CODE', contains=['CODE'])
DX = DX[['SUBJECT_ID', 'HADM_ID', C_DXCODE]].rename(columns={C_DXCODE: 'CODE'}).dropna(subset=['CODE'])
DX['CODE'] = DX.CODE.astype(str).str.strip()
DX = DX.drop_duplicates(['HADM_ID', 'CODE'])
CODES_BY_ADM = DX.groupby('HADM_ID').CODE.agg(set)
INC_ADM = set(DX[DX.CODE.isin(set(LIST_A) | set(LIST_B))].HADM_ID) & set(ADM.index)
EXC_ADM = set(DX[DX.CODE.isin(set(LIST_C)) | DX.CODE.str.contains(EXCL_PREFIX, regex=True)].HADM_ID)


def child_group(codes):
    gs = {INC_GROUP[c] for c in codes if c in INC_GROUP}
    return next((g for g in GROUP_ORDER if g in gs), None)


FLOW = {}


def base_cohort():
    """Index admissions up to the age criteria (the test criterion depends on the time anchor)."""
    steps = []
    A = ADM.loc[sorted(INC_ADM)]
    steps.append(('Admissions with an inclusion code', len(A), A.SUBJECT_ID.nunique()))
    A = A.sort_values(['SUBJECT_ID', 'ADMITTIME', 'HADM_ID']).drop_duplicates('SUBJECT_ID')
    steps.append(('First such admission per child (index admission)', len(A), A.SUBJECT_ID.nunique()))
    A = A[~A.index.isin(EXC_ADM)]
    steps.append(('No exclusion code in the index admission', len(A), A.SUBJECT_ID.nunique()))
    A = A[A.AGE_DAYS.ge(AGE_MIN_DAYS)]
    steps.append(('Aged 28 days or older', len(A), A.SUBJECT_ID.nunique()))
    A = A[A.AGE_DAYS.lt(AGE_MAX_YEARS * 365.25)]
    steps.append(('Younger than 18 years', len(A), A.SUBJECT_ID.nunique()))
    A = A.copy()
    A['CODES'] = A.index.map(CODES_BY_ADM)
    A['GROUP'] = A.CODES.map(child_group)
    A['SENS_A_DROP'] = A.CODES.map(lambda s: bool(s & SENS_A_DROP))
    A['LIST_A'] = A.CODES.map(lambda s: bool(s & set(LIST_A)))
    return A, steps


BASE, BASE_STEPS = base_cohort()
CAND_ADM = set(BASE.index)
CAND_SUBJ = set(BASE.SUBJECT_ID)
print('Index admissions selected (counts are in the cohort flow below).')

Index admissions selected (counts are in the cohort flow below).


## Laboratory results and vital signs
Pre-specified items (fixed after the outcome-blind review of the laboratory dictionary), converted to the Wuhan unit. Censored
results (for example "<0.05") enter with the reporting limit, as in Wuhan. A result filed without an admission number is linked to
the admission when it lies within 24 h before it or during it. Specimen time (CHARTTIME) orders the results. Vital signs come from
CHARTEVENTS: temperature, heart rate (monitor heart rate where charted at a time, otherwise the pulse) and respiratory rate,
with the Wuhan plausibility ranges.

In [6]:
LAB_ITEMS = {  # concept: {ITEMID: English label in the PIC dictionary}
    'neutrophils_abs': {'5094': 'Neutrophils'}, 'wbc': {'5141': 'WBC Count'}, 'lymphocytes_pct': {'5111': 'Lymphocytes, Percent'},
    'platelets': {'5129': 'Platelet Count'}, 'hemoglobin': {'5099': 'Hemoglobin'},
    'hs_crp': {'5626': 'C-Reactive Protein', '5821': 'C-Reactive Protein'}, 'procalcitonin': {'6085': 'Procalcitonin'},
    'alt': {'5026': 'Alanine Aminotransferase (ALT)', '5195': 'Alanine Aminotransferase (ALT)'},
    'ast': {'5031': 'Asparate Aminotransferase (AST)'}, 'ldh': {'5057': 'Lactate Dehydrogenase (LD)'}, 'ck': {'5038': 'Creatine Kinase (CK)'},
    'bilirubin_total': {'5075': 'Bilirubin, Total'}, 'albumin': {'5024': 'Albumin', '6465': 'Serum Albumin'},
    'creatinine': {'5032': 'Creatinine', '5041': 'Creatinine'}, 'urea': {'5033': 'Urea', '6477': 'Urea'},
    'sodium': {'5230': 'Sodium, Whole Blood'}, 'ammonia': {'6334': 'Ammonia'}, 'lactate': {'5227': 'Lactate'},
    'inr': {'5174': 'INR(PT)'}, 'aptt': {'5161': 'PTT'}, 'fibrinogen': {'5164': 'Fibrinogen, Functional'}, 'd_dimer': {'5163': 'D-Dimer'},
    'il6': {'5604': 'human IL-6'}, 'il10': {'5601': 'human IL-10'}, 'tnf_alpha': {'5620': 'human TNF'}, 'ifn_gamma': {'5600': 'human IFN-r'},
    'ferritin': {'6161': 'Ferritin', '6459': 'Ferritin'}}
ITEM_CONCEPT = {it: c for c, d in LAB_ITEMS.items() for it in d}
VITAL_ITEMS = {'1001': ('temp_c', 'Temperature'), '1003': ('heart_rate_monitor', 'Heart Rate'), '1002': ('pulse', 'Pulse'),
               '1004': ('resp_rate', 'Respiratory Rate')}
VITAL_RANGE = {'temp_c': (30, 43.5), 'heart_rate_monitor': (20, 300), 'pulse': (20, 300), 'resp_rate': (4, 150)}   # Wuhan freeze


def norm_unit(u):
    if not isinstance(u, str) or not u.strip():
        return ''
    u = u.strip().lower().replace('μ', 'u').replace('µ', 'u').replace('×', '').replace(' ', '').replace('（', '(').replace('）', ')')
    u = re.sub(r'^(x|\*)', '', u)
    u = re.sub(r'10(\^|\*|e|~)?9/l|g/l\(10\^9\)|10\^9l', '10^9/l', u) if re.search(r'10(\^|\*|e|~)?9', u) else u
    u = u.replace('iu/l', 'u/l').replace('sec', 's').replace('秒', 's').replace('meq/l', 'mmol/l')
    return u


CONV = {  # normalized unit -> factor to the Wuhan unit (as in PIC_SUB_01)
    'neutrophils_abs': {'10^9/l': 1, '10^3/ul': 1, '/nl': 1, 'g/l': 1}, 'wbc': {'10^9/l': 1, '10^3/ul': 1, '/nl': 1, 'g/l': 1},
    'platelets': {'10^9/l': 1, '10^3/ul': 1, '/nl': 1, 'g/l': 1}, 'lymphocytes_pct': {'%': 1},
    'hemoglobin': {'g/l': 1, 'g/dl': 10}, 'hs_crp': {'mg/l': 1, 'mg/dl': 10}, 'procalcitonin': {'ng/ml': 1, 'ug/l': 1},
    'alt': {'u/l': 1}, 'ast': {'u/l': 1}, 'ldh': {'u/l': 1}, 'ck': {'u/l': 1},
    'bilirubin_total': {'umol/l': 1, 'mg/dl': 17.1}, 'albumin': {'g/l': 1, 'g/dl': 10}, 'creatinine': {'umol/l': 1, 'mg/dl': 88.42},
    'urea': {'mmol/l': 1}, 'sodium': {'mmol/l': 1}, 'ammonia': {'umol/l': 1, 'ug/dl': 0.587}, 'lactate': {'mmol/l': 1, 'mg/dl': 0.111},
    'inr': {'': 1, 'ratio': 1, 'inr': 1}, 'aptt': {'s': 1}, 'fibrinogen': {'g/l': 1, 'mg/dl': 0.01},
    'd_dimer': {'mg/lfeu': 1, 'mg/l': 1, 'ug/ml': 1, 'ug/mlfeu': 1, 'ug/l': 0.001, 'ng/ml': 0.001, 'ug/lfeu': 0.001, 'ng/mlfeu': 0.001},
    'il6': {'pg/ml': 1, 'ng/l': 1}, 'il10': {'pg/ml': 1, 'ng/l': 1}, 'tnf_alpha': {'pg/ml': 1, 'ng/l': 1}, 'ifn_gamma': {'pg/ml': 1, 'ng/l': 1},
    'ferritin': {'ng/ml': 1, 'ug/l': 1}}

# Item check against the dictionary (English label must match; the run stops otherwise).
DLAB = read('D_LABITEMS')
DLAB = pd.DataFrame({'ITEMID': DLAB[pick(DLAB.columns, 'ITEMID')].astype(str).str.strip(),
                     'LABEL': DLAB[pick(DLAB.columns, 'LABEL')].astype(str).str.strip(),
                     'LABEL_CN': DLAB[pick(DLAB.columns, 'LABEL_CN', contains=['LABEL', 'CN'])] if pick(DLAB.columns, 'LABEL_CN', contains=['LABEL', 'CN']) else ''}
                    ).drop_duplicates('ITEMID').set_index('ITEMID')
_bad = [(it, lab, DLAB.LABEL.get(it)) for c, d in LAB_ITEMS.items() for it, lab in d.items()
        if it not in DLAB.index or DLAB.LABEL[it].strip().lower() != lab.lower()]
DIT = read('D_ITEMS')
DIT = pd.DataFrame({'ITEMID': DIT[pick(DIT.columns, 'ITEMID')].astype(str).str.strip(),
                    'LABEL': DIT[pick(DIT.columns, 'LABEL')].astype(str).str.strip()}).drop_duplicates('ITEMID').set_index('ITEMID')
_bad += [(it, lab, DIT.LABEL.get(it)) for it, (_, lab) in VITAL_ITEMS.items() if it not in DIT.index or DIT.LABEL[it].lower() != lab.lower()]
if _bad:
    raise RuntimeError(f'Pre-specified items not found with the expected label (item, expected, found): {_bad}')

# Laboratory results of the candidate admissions (and of their children without an admission number).
NUM = re.compile(r'^\s*(<=|>=|<|>|≤|≥)?\s*([-+]?\d+(?:\.\d+)?)\s*$')
unit_counts, keep = {}, []
for ch in read('LABEVENTS', ['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM', 'VALUEUOM'], chunksize=CHUNK):
    ch['ITEMID'] = ch.ITEMID.astype(str).str.strip()
    ch = ch[ch.ITEMID.isin(ITEM_CONCEPT)]
    if ch.empty:
        continue
    for (it, u), k in ch.groupby(['ITEMID', ch.VALUEUOM.fillna('')]).size().items():
        unit_counts.setdefault(it, {}).setdefault(u, 0)
        unit_counts[it][u] += int(k)
    sel = ch.HADM_ID.isin(CAND_ADM) | (ch.HADM_ID.isna() & ch.SUBJECT_ID.isin(CAND_SUBJ))
    if sel.any():
        keep.append(ch[sel])
LAB = pd.concat(keep, ignore_index=True) if keep else pd.DataFrame(columns=['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM', 'VALUEUOM'])
LAB['T'] = to_dt(LAB.CHARTTIME)
LAB['V'] = pd.to_numeric(LAB.VALUENUM, errors='coerce')
_p = LAB.VALUE.astype(str).str.extract(NUM)
LAB['CENSORED'] = LAB.V.isna() & _p[0].notna()
LAB['V'] = LAB.V.fillna(pd.to_numeric(_p[1], errors='coerce'))
_sym = _p[0].fillna('')
LAB['CDIR'] = np.where(_sym.isin(['<', '<=', '≤']), 'low', np.where(_sym.isin(['>', '>=', '≥']), 'high', ''))
ITEM_UNIT = {it: max(((u, k) for u, k in d.items() if u), key=lambda x: x[1], default=('', 0))[0] for it, d in unit_counts.items()}
LAB['UNIT_N'] = [norm_unit(u if isinstance(u, str) and u else ITEM_UNIT.get(it, '')) for u, it in zip(LAB.VALUEUOM, LAB.ITEMID)]
LAB['concept'] = LAB.ITEMID.map(ITEM_CONCEPT)
LAB['FACTOR'] = [CONV[c].get(u) for c, u in zip(LAB.concept, LAB.UNIT_N)]
LAB['VC'] = LAB.V * pd.to_numeric(LAB.FACTOR, errors='coerce')          # value in the Wuhan unit (NaN if the unit is not convertible)
LAB['LINKED'] = False
_noh = LAB.HADM_ID.isna()
if _noh.any():
    _c = BASE[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME']].reset_index(drop=True)
    _l = LAB[_noh].reset_index().merge(_c, on='SUBJECT_ID', how='inner', suffixes=('', '_A'))
    _l = _l[(_l['T'] >= _l.ADMITTIME - pd.Timedelta(hours=24)) & (_l['T'] <= _l.DISCHTIME)]
    _l = _l.sort_values(['index', 'ADMITTIME']).drop_duplicates('index')
    LAB.loc[_l['index'].values, 'HADM_ID'] = _l['HADM_ID_A'].values
    LAB.loc[_l['index'].values, 'LINKED'] = True
LAB = LAB[LAB.HADM_ID.isin(CAND_ADM) & LAB.V.notna() & LAB['T'].notna()].copy()
print(f'Laboratory rows for the candidate admissions: {C(len(LAB)).share} (linked without an admission number: {C(int(LAB.LINKED.sum())).share}; '
      f'unit not convertible: {C(int(LAB.VC.isna().sum())).share})')

# Vital signs.
vkeep = []
for ch in read('CHARTEVENTS', ['HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM'], chunksize=CHUNK):
    ch['ITEMID'] = ch.ITEMID.astype(str).str.strip()
    ch = ch[ch.ITEMID.isin(VITAL_ITEMS) & ch.HADM_ID.isin(CAND_ADM)]
    if not ch.empty:
        vkeep.append(ch)
VIT = pd.concat(vkeep, ignore_index=True) if vkeep else pd.DataFrame(columns=['HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM'])
VIT['T'] = to_dt(VIT.CHARTTIME)
VIT['V'] = pd.to_numeric(VIT.VALUENUM, errors='coerce').fillna(pd.to_numeric(VIT.VALUE, errors='coerce'))
VIT['var'] = VIT.ITEMID.map(lambda i: VITAL_ITEMS[i][0])
_t = VIT['var'].eq('temp_c') & VIT.V.between(86, 110)                  # Fahrenheit, if any, to Celsius
VIT.loc[_t, 'V'] = (VIT.loc[_t, 'V'] - 32) / 1.8
_lo = VIT['var'].map(lambda v: VITAL_RANGE[v][0])
_hi = VIT['var'].map(lambda v: VITAL_RANGE[v][1])
VIT_IMPLAUSIBLE = int((VIT.V.notna() & ~VIT.V.between(_lo, _hi)).sum())
VIT = VIT[VIT.V.notna() & VIT['T'].notna() & VIT.V.between(_lo, _hi)].copy()
print(f'Vital-sign rows: {C(len(VIT)).share} (implausible values set aside: {C(VIT_IMPLAUSIBLE).share})')

Laboratory rows for the candidate admissions: 49380 (linked without an admission number: 0; unit not convertible: 0)
Vital-sign rows: 22440 (implausible values set aside: 5)


In [7]:
# First value of every measurement per admission, for a time anchor (hospital admission, or first ICU entry for S-d).
ANCHORS = {'admission': BASE.ADMITTIME, 'icu_entry': BASE.ICU_IN}


def first_values(anchor):
    """First values (Wuhan unit) and, for the laboratory tests, the censoring of each first value ('low', 'high' or '')."""
    a = ANCHORS[anchor]
    res, cen = {}, {}
    d = LAB[LAB.VC.notna()].copy()
    d['H'] = (d['T'] - d.HADM_ID.map(a)).dt.total_seconds() / 3600
    for c in LAB22 + VALIDATORS:
        lo, hi = VALIDATOR_WINDOW_H if c in VALIDATORS else LAB_WINDOW_H
        w = d[d.concept.eq(c) & d.H.between(lo, hi)].sort_values(['HADM_ID', 'T', 'ITEMID']).drop_duplicates('HADM_ID').set_index('HADM_ID')
        res[c], cen[c] = w.VC, w.CDIR
    v = VIT.copy()
    v['H'] = (v['T'] - v.HADM_ID.map(a)).dt.total_seconds() / 3600
    v = v[v.H.between(*VITAL_WINDOW_H)]
    for c in ['temp_c', 'resp_rate']:
        res[c] = v[v['var'].eq(c)].sort_values(['HADM_ID', 'T']).drop_duplicates('HADM_ID').set_index('HADM_ID').V
    hr = v[v['var'].isin(['heart_rate_monitor', 'pulse'])].copy()
    hr['pref'] = hr['var'].ne('heart_rate_monitor').astype(int)          # at the same time, the monitor heart rate first
    res['heart_rate'] = hr.sort_values(['HADM_ID', 'T', 'pref']).drop_duplicates('HADM_ID').set_index('HADM_ID').V
    F = pd.DataFrame(res).reindex(index=BASE.index)
    for c in LAB22 + VALIDATORS + ['temp_c', 'heart_rate', 'resp_rate']:
        if c not in F:
            F[c] = np.nan
    FC = pd.DataFrame(cen).reindex(index=BASE.index).fillna('')
    return F, FC


FV, FCV = {}, {}
for _k in ANCHORS:
    FV[_k], FCV[_k] = first_values(_k)
NTESTS = {k: FV[k][LAB22].notna().sum(axis=1) for k in ANCHORS}


def eligible(anchor):
    A = BASE if anchor == 'admission' else BASE[BASE.ICU_IN.notna()]
    return A[NTESTS[anchor].reindex(A.index).fillna(0).ge(MIN_TESTS)].copy()


COH = eligible('admission')
COH_ICU = eligible('icu_entry')
N_ELIG = len(COH)
GATE_OK = N_ELIG >= GATE_N
FLOW_VALS = [BASE_STEPS[0][1]] + [st_[2] for st_ in BASE_STEPS[1:]] + [N_ELIG]     # admissions, then children
FLOW_DESC = [st_[0] for st_ in BASE_STEPS] + [f'At least {MIN_TESTS} of the 22 blood tests from 24 h before to 24 h after admission (eligible)']
FLOW_UNIT = ['admissions'] + ['children'] * (len(FLOW_VALS) - 1)
flow_rows = [{'step': i + 1, 'description': d_, 'unit': u_, 'count_n': C(v_)} for i, (d_, u_, v_) in enumerate(zip(FLOW_DESC, FLOW_UNIT, FLOW_VALS))]
save(flow_rows, '02_flow.csv')
share_print(flow_rows)
print(f'\nEligible children (rounded to 5): {r5(N_ELIG)}. Registered threshold of {GATE_N}: {"met" if GATE_OK else "NOT met"}.')

step                                                                           description       unit count_n
   1                                                     Admissions with an inclusion code admissions     370
   2                                      First such admission per child (index admission)   children     360
   3                                              No exclusion code in the index admission   children     355
   4                                                                 Aged 28 days or older   children     350
   5                                                                 Younger than 18 years   children     350
   6 At least 11 of the 22 blood tests from 24 h before to 24 h after admission (eligible)   children     245

Eligible children (rounded to 5): 245. Registered threshold of 200: met.


In [8]:
# Cohort description, code groups, items and availability (descriptive; no subphenotypes).
GRP_COUNT = COH.GROUP.value_counts().reindex(GROUP_ORDER).fillna(0).astype(int)
grp_rows = [{'code_group': g, 'children_n': C(GRP_COUNT[g]), 'children_pct': pct_txt(int(GRP_COUNT[g]), N_ELIG)} for g in GROUP_ORDER]
save(grp_rows, '03_code_groups.csv')

N_DIED = int(COH.DIED.sum())
_male, _picu = int(COH.SEX.eq('male').sum()), int(COH.FIRST_UNIT.eq('PICU').sum())
desc = {'children_n': C(N_ELIG), 'age_years_median_iqr': q(COH.AGE_DAYS / 365.25),
        'male_n': C(_male), 'male_pct': pct_txt(_male, N_ELIG),
        'in_hospital_deaths_n': C(N_DIED), 'in_hospital_deaths_pct': pct_txt(N_DIED, N_ELIG),
        'first_icu_unit_picu_n': C(_picu), 'first_icu_unit_picu_pct': pct_txt(_picu, N_ELIG),
        'blood_tests_of_22_median_iqr': q(NTESTS['admission'].reindex(COH.index)),
        'missing_pct_24_indicators': None}
F0, FC0 = FV['admission'].reindex(COH.index), FCV['admission'].reindex(COH.index)
item_rows = []
for c, d in LAB_ITEMS.items():
    for it in d:
        x = LAB[LAB.ITEMID.eq(it) & LAB.HADM_ID.isin(set(COH.index))]
        units = '; '.join(f'{u or "(none)"}' for u in sorted(set(x.UNIT_N)))
        item_rows.append({'concept': c, 'itemid': it, 'label': d[it], 'rows_eligible_children_n': C(len(x)),
                          'units_normalized': units, 'rows_unit_not_convertible_n': C(int(x.VC.isna().sum())),
                          'rows_censored_n': C(int(x.CENSORED.sum())),
                          'censored_pct': pct_txt(int(x.CENSORED.sum()), len(x)) if len(x) else Txt(None),
                          'linked_without_admission_number_n': C(int(x.LINKED.sum()))})
save(item_rows, '04_items.csv')
av_rows = []
for c in LAB22 + ['temp_c', 'heart_rate', 'resp_rate'] + VALIDATORS:
    _sv = F0[c]
    wr_ = WREF['indicators'].get(c, {})
    av_rows.append({'measure': LABEL.get(c, c), 'in_model': c in IND or c in ('heart_rate', 'resp_rate'),
                    'measured_n': C(int(_sv.notna().sum())), 'measured_pct': pct_txt(int(_sv.notna().sum()), N_ELIG),
                    'wuhan_measured_pct': wr_.get('pct_measured'), 'pic_median_iqr_wuhan_unit': q(_sv),
                    'wuhan_median_iqr': f"{wr_['median']:.3g} ({wr_['p25']:.3g}-{wr_['p75']:.3g})" if wr_ else None})
save(av_rows, '05_availability.csv')

,measure,in_model,measured_n,measured_pct,wuhan_measured_pct,pic_median_iqr_wuhan_unit,wuhan_median_iqr
0,Neutrophil count,True,235,96.7,94.0,6.17 (3.95-9.86),5.09 (2.63-8.73)
1,White cell count,False,235,96.7,94.1,8.77 (6.52-12.7),8.03 (5.23-11.8)
2,Lymphocyte percentage,True,232,95.5,86.2,18.4 (11.5-31.2),23.9 (12.7-40.2)
3,Platelets,True,235,96.7,94.0,257 (162-359),250 (169-331)
4,Hemoglobin,True,235,96.7,94.0,118 (106-128),116 (106-126)
5,High-sensitivity CRP,True,235,96.7,84.3,5 (1-19),5.63 (0.983-21.6)
6,Procalcitonin,True,134,55.1,89.5,0.326 (0.064-11.9),0.43 (0.12-5.21)
7,ALT,True,220,90.5,99.4,20 (12-45.5),23 (15-39)
8,AST,True,208,85.6,99.4,35.5 (27-68.2),44 (30-73.5)
9,LDH,True,220,90.5,98.4,314 (236-450),296 (241-475)


## Model application
Each laboratory value (Wuhan unit) is divided by the effective Wuhan reference limit for the child's age band. Heart and
respiratory rates become z-scores for age (Fleming 2011), as in Wuhan. Then come the natural log for the indicators
log-transformed in Wuhan, the Wuhan winsorizing limits and the Wuhan means and SDs. The posterior class probabilities come from the frozen six-class model; a
missing indicator is left out of the likelihood. Each child is assigned to the most likely class.

**hs-CRP reporting floor.** The Wuhan laboratory reported hs-CRP below about 0.8 mg/L as below a limit (23% of first values),
and these results entered at the limit. The least abnormal class S6 sits at that point: 92% of its children had a censored
hs-CRP, and its hs-CRP SD on the standardized scale is 0.02. PIC reports hs-CRP below 1 mg/L as "<1". Entered at 1 mg/L, such a
result would lie about 5 class SDs above S6, and no child with it could be assigned to S6. So an hs-CRP result reported as below a
limit, or lying below Wuhan's limit, enters at Wuhan's limit (ratio 0.8/3 to the reference limit). The rule applies to any test
that Wuhan reported as below a limit in at least 5% of first values, which is only hs-CRP. Sensitivity analysis S-g enters these
results at the PIC limit instead.

In [9]:
def band_of(age_days):
    for name, lo, hi in BAND_EDGES:
        if lo <= age_days < hi:
            return name
    return None


FZ = norm.ppf(np.array(FLEM['centiles'], float) / 100)
FBANDS = FLEM['bands_months']


def age_z(values, age_days, table):
    """z-score for age (Wuhan definition): piecewise-linear interpolation between the Fleming centiles on the standard-normal scale;
    beyond the 1st or 99th centile, linear extrapolation with the slope of the outer interval."""
    out = np.full(len(values), np.nan)
    for i, (v, a) in enumerate(zip(np.asarray(values, float), np.asarray(age_days, float) / 30.4375)):
        if np.isnan(v) or np.isnan(a):
            continue
        b = next((j for j, (lo, hi) in enumerate(FBANDS) if lo <= a < hi), len(FBANDS) - 1)
        c = np.asarray(table[b], float)
        if v <= c[0]:
            out[i] = FZ[0] + (v - c[0]) * (FZ[1] - FZ[0]) / (c[1] - c[0])
        elif v >= c[-1]:
            out[i] = FZ[-1] + (v - c[-1]) * (FZ[-1] - FZ[-2]) / (c[-1] - c[-2])
        else:
            out[i] = np.interp(v, c, FZ)
    return out


FLOORS = {c: f['floor_ratio'] for c, f in PKG['reporting_floors'].items() if f['applied']}    # hs-CRP only


def model_input(F, A, FC=None, floors=True):
    """Indicators on the model's raw scale for the children in A. With floors, a test with a Wuhan reporting floor (hs-CRP) enters
    at Wuhan's floor when its result is reported as below a limit or lies below the floor."""
    R = pd.DataFrame(index=A.index)
    bands = A.AGE_DAYS.map(band_of)
    for c in MODEL_LABS:
        lim = bands.map(lambda b: REFLIM[c]['bands'][b]['limit']).astype(float)
        R[c] = F[c].reindex(A.index).astype(float) / lim
        if floors and c in FLOORS:
            low = (FC[c].reindex(A.index).eq('low') if FC is not None else False) | (R[c] < FLOORS[c])
            R.loc[low & R[c].notna(), c] = FLOORS[c]
    R['temp_c'] = F['temp_c'].reindex(A.index).astype(float)
    R['hr_z'] = age_z(F['heart_rate'].reindex(A.index).values, A.AGE_DAYS.values, FLEM['heart_rate'])
    R['rr_z'] = age_z(F['resp_rate'].reindex(A.index).values, A.AGE_DAYS.values, FLEM['resp_rate'])
    return R[IND]


def log_scale(R):
    X = R.astype(float).copy()
    with np.errstate(divide='ignore', invalid='ignore'):
        for c in IND:
            if PRE[c]['log']:
                X[c] = np.log(X[c])
    return X


def standardize(R, pre=None):
    """Log scale, winsorizing and standardization with the frozen Wuhan constants (or with constants given in pre)."""
    pre = pre or PRE
    X = log_scale(R)
    Z = pd.DataFrame(index=R.index)
    for c in IND:
        p = pre[c]
        Z[c] = (X[c].clip(p['winsor_low'], p['winsor_high']) - p['mean']) / p['sd']
    return Z


def internal_constants(R):
    """PIC's own constants (sensitivity analysis S-c): 0.5th and 99.5th percentiles, mean and SD within the analysed children."""
    X = log_scale(R)
    pre = {}
    for c in IND:
        x = X[c][np.isfinite(X[c])]
        lo, hi = float(x.quantile(0.005)), float(x.quantile(0.995))
        xw = X[c].clip(lo, hi)
        pre[c] = {'log': PRE[c]['log'], 'winsor_low': lo, 'winsor_high': hi, 'mean': float(xw.mean()), 'sd': float(xw.std(ddof=1))}
    return pre


def posterior(Z, w=None, mu=None, var=None):
    """Posterior class probabilities of the diagonal Gaussian mixture; missing values are left out (full-information likelihood).
    Returns the posteriors, the log-likelihood per child and the number of observed indicators per child."""
    w = W if w is None else w
    mu = MU if mu is None else mu
    var = VAR if var is None else var
    X = np.asarray(Z, float)
    M = (~np.isnan(X)).astype(float)
    X0 = np.where(M > 0, X, 0.0)
    prec = 1.0 / var
    quad = (X0 ** 2) @ prec.T - 2 * X0 @ (mu * prec).T + M @ (mu ** 2 * prec).T
    lp = -0.5 * (M @ np.log(2 * np.pi * var).T + quad) + np.log(w)[None, :]
    lse = logsumexp(lp, axis=1)
    return np.exp(lp - lse[:, None]), lse, M.sum(axis=1)


def profile_r(Zv, members, k=S1):
    """Pearson correlation across indicators between the mean standardized profile of the members and the Wuhan class-k profile."""
    if members.sum() == 0:
        return np.nan, None
    with np.errstate(all='ignore'), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p = np.nanmean(Zv[members], axis=0)
    ok = np.isfinite(p)
    if ok.sum() < 3:
        return np.nan, p
    return float(np.corrcoef(p[ok], MU[k][ok])[0, 1]), p


def boot_r(Zv, modal, r_hat, k=S1, seed=SEED):
    """95% CI of the profile correlation: normal approximation on Fisher's z scale, centred on the estimate, with the SD of z over
    2,000 bootstrap resamples of children (a percentile interval would sit below the estimate, because resampling adds noise to the
    mean profile)."""
    rng = np.random.RandomState(seed)
    n, out = len(modal), []
    for _ in range(N_BOOT):
        i = rng.randint(0, n, n)
        m = modal[i] == k
        if m.sum() < 2:
            continue
        r, _ = profile_r(Zv[i], m, k)
        if np.isfinite(r) and abs(r) < 1:
            out.append(np.arctanh(r))
    if len(out) < 100 or not np.isfinite(r_hat) or abs(r_hat) >= 1:
        return np.nan, np.nan, len(out)
    zc, sd = np.arctanh(r_hat), float(np.std(out, ddof=1))
    return float(np.tanh(zc - 1.96 * sd)), float(np.tanh(zc + 1.96 * sd)), len(out)


def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    labs = np.union1d(a, b)
    po = float(np.mean(a == b))
    pe = float(sum(np.mean(a == l) * np.mean(b == l) for l in labs))
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


def log_validator(v):
    """Log scale; a result of exactly 0 enters as half the smallest positive result of that test (Wuhan rule)."""
    v = pd.Series(v, dtype=float)
    pos = v[v > 0]
    half = pos.min() / 2 if len(pos) else np.nan
    return np.log(v.where(v.isna() | (v > 0), half)).values


def gm_ratio(lv, g1, g0, seed=SEED):
    """Ratio of geometric means (group 1 / group 0) with a percentile-bootstrap 95% CI (children resampled). Needs at least 5
    measured children in each group."""
    ok = np.isfinite(lv)
    n1, n0 = int((g1 & ok).sum()), int((g0 & ok).sum())
    if n1 < 5 or n0 < 5:
        return n1, n0, None, None, None
    e = float(np.exp(lv[g1 & ok].mean() - lv[g0 & ok].mean()))
    rng = np.random.RandomState(seed)
    bs = []
    for _ in range(N_BOOT):
        i = rng.randint(0, len(lv), len(lv))
        a, b = lv[i][g1[i] & ok[i]], lv[i][g0[i] & ok[i]]
        if len(a) >= 2 and len(b) >= 2:
            bs.append(a.mean() - b.mean())
    return n1, n0, e, float(np.exp(np.percentile(bs, 2.5))), float(np.exp(np.percentile(bs, 97.5)))


def risk_ratio(a, n1, c, n0):
    """Risk ratio with the Katz log 95% CI; None when not estimable (no events in a group, or an empty group)."""
    if not n1 or not n0:
        return None, None, None
    if a == 0 or c == 0:
        return ((a / n1) / (c / n0) if c else np.inf), None, None
    rr = (a / n1) / (c / n0)
    se = np.sqrt(1 / a - 1 / n1 + 1 / c - 1 / n0)
    return rr, rr * np.exp(-1.96 * se), rr * np.exp(1.96 * se)


def rcs3(x, knots):
    t1, t2, t3 = knots
    p = lambda u: np.clip(u, 0, None) ** 3
    return (p(x - t1) - p(x - t2) * (t3 - t1) / (t3 - t2) + p(x - t3) * (t2 - t1) / (t3 - t2)) / (t3 - t1) ** 2


def logistic(y, X, max_iter=100):
    """Maximum-likelihood logistic regression (Newton-Raphson). Returns coefficients, their covariance and convergence."""
    b = np.zeros(X.shape[1])
    converged = False
    for _ in range(max_iter):
        p = 1 / (1 + np.exp(-np.clip(X @ b, -30, 30)))
        H = X.T @ (X * (p * (1 - p))[:, None])
        try:
            step = np.linalg.solve(H, X.T @ (y - p))
        except np.linalg.LinAlgError:
            return b, None, False
        b = b + step
        if np.max(np.abs(step)) < 1e-9:
            converged = True
            break
    ok = bool(converged and np.all(np.isfinite(b)) and np.max(np.abs(b)) < 15)
    try:
        cov = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        cov, ok = None, False
    return b, cov, ok


def rule_positive(F, A):
    """S1 bedside rule (frozen): at least 2 of AST >= 200 U/L, LDH >= 700 U/L, INR >= 1.5; assessable when AST and LDH are measured."""
    crit = [(F[c['test']].reindex(A.index) >= c['value']).values for c in RULE['criteria']]
    pos = np.sum(crit, axis=0) >= RULE['positive_if_at_least']
    assess = (F['ast'].reindex(A.index).notna() & F['ldh'].reindex(A.index).notna()).values
    return assess, pos

## Primary outcome: does the hyperinflammatory subphenotype (S1) replicate?
Pre-specified criterion: at least 10 PIC children are assigned to S1, and the Pearson correlation across the 24 indicators between
their mean standardized profile and the Wuhan S1 profile (model means) is at least 0.80. The correlation's 95% CI uses Fisher's z
with the SD from 2,000 bootstrap resamples of children. For comparison, the Wuhan S1 profile correlates 0.51 with the next class
(S2) and negatively with the other four.

In [10]:
RESULTS = {}


def analyse(label, A, F, FC, pre=None, drop=(), floors=True, compare=False):
    """Apply the frozen model to the children in A (first values F, censoring FC) and compute the pre-specified summary measures."""
    R = model_input(F, A, FC, floors)
    for c in drop:
        R[c] = np.nan
    pre = internal_constants(R) if pre == 'internal' else pre
    Z = standardize(R, pre)
    Zv = Z.values
    post, lse, nobs = posterior(Zv)
    modal = post.argmax(1)
    n = len(A)
    s1 = modal == S1
    n_s1 = int(s1.sum())
    r, prof = profile_r(Zv, s1)
    r_lo, r_hi, r_b = boot_r(Zv, modal, r) if n_s1 >= 2 else (np.nan, np.nan, 0)
    replicated = bool(n_s1 >= MIN_S1 and np.isfinite(r) and r >= MIN_R)
    died = A.DIED.values.astype(bool)
    a_, n1, c_, n0 = int(died[s1].sum()), n_s1, int(died[~s1].sum()), n - n_s1
    rr = risk_ratio(a_, n1, c_, n0)
    lv = {v: log_validator(F[v].reindex(A.index)) for v in VALIDATORS}
    il6 = gm_ratio(lv['il6'], s1, modal == S6)
    il10 = gm_ratio(lv['il10'], s1, modal == S6)
    out = {'label': label, 'n': n, 'n_s1': n_s1, 'r': r, 'r_ci': (r_lo, r_hi), 'replicated': replicated, 'death': (a_, n1, c_, n0),
           'rr': rr, 'il6': il6, 'il10': il10, 'modal': modal, 'post': post, 'Z': Z, 'R': R, 'lse': lse, 'nobs': nobs, 'prof': prof,
           'lv': lv, 'index': A.index, 'compare': compare}
    return out


def death_cells(res):
    """S1 deaths, S1 survivors, deaths in the other classes, survivors in the other classes."""
    a_, n1, c_, n0 = res['death']
    return a_, n1 - a_, c_, n0 - c_


def r_text(res):
    return fmt_ci(res['r'], *res['r_ci'], d=3) if np.isfinite(res['r']) else None


def r_share(res):
    """Profile correlation and CI, 2 decimals (shareable form)."""
    r, (lo, hi) = res['r'], res['r_ci']
    if not np.isfinite(r):
        return None
    return f'{r:.2f} ({lo:.2f}-{hi:.2f})' if np.isfinite(lo) else f'{r:.2f} (NA)'


def summary_row(res, kind, parent=None):
    """One row of the sensitivity table. kind: 'primary', 'subset' (children of the primary analysis), 'same' (the same children,
    other preprocessing) or 'other' (another population); the death risk ratio is in the local tables only."""
    rr, lo, hi = res['rr']
    rt = r_text(res)
    na_v = 'not estimable (fewer than 5 measured in a group)'
    row = {'analysis': res['label'], 'children_n': C(res['n']), 'S1_n': C(res['n_s1']), 'S1_pct': pct_txt(res['n_s1'], res['n']),
           'S1_profile_r_ci': est(rt, r_share(res), hide=res['n_s1'] < 5) if rt else Txt(None),
           'replication_criterion_met': 'yes' if res['replicated'] else 'no',
           'il6_ratio_S1_vs_S6': ci_est(*res['il6'][2:], na=na_v), 'il10_ratio_S1_vs_S6': ci_est(*res['il10'][2:], na=na_v),
           'death_rr_S1_vs_others': ci_local(rr, lo, hi) if (rr is not None and lo is not None) else Txt('not estimable', '')}
    if res.get('compare') and parent is not None:
        common = res['index'].intersection(parent['index'])
        if len(common):
            m1 = pd.Series(res['modal'], index=res['index']).loc[common]
            m0 = pd.Series(parent['modal'], index=parent['index']).loc[common]
            row['kappa_vs_primary'] = round(kappa(m1.values, m0.values), 2)
            row['common_children_n'] = C(len(common))
    return row


if GATE_OK:
    PRIM = analyse('Primary', COH, F0, FC0)
    RESULTS['primary'] = PRIM
    SIZES = np.bincount(PRIM['modal'], minlength=K)
    print(f"S1: {r5(SIZES[S1])} of {r5(N_ELIG)} children (rounded to 5); profile r = "
          f"{r_share(PRIM) if PRIM['n_s1'] >= 5 else SUP}; pre-specified criterion {'MET' if PRIM['replicated'] else 'NOT met'}")
    desc['missing_pct_24_indicators'] = round(100 * float(np.isnan(PRIM['Z'].values).mean()), 1)
else:
    print('Registered threshold not met: the model is not applied (registered decision P15).')

S1: 25 of 245 children (rounded to 5); profile r = 0.94 (0.90-0.97); pre-specified criterion MET


## Secondary outcomes (estimates with 95% CIs)
Class sizes and certainty; class profiles; IL-6 and IL-10 (first value from 24 h before to 48 h after admission; TNF-alpha,
IFN-gamma and ferritin as further validators); in-hospital death by subphenotype; the S1 bedside rule; and the model's fit in PIC.

In [11]:
if GATE_OK:
    P = PRIM
    modal, post, Zv = P['modal'], P['post'], P['Z'].values
    n = P['n']
    sizes = SIZES
    cls_rows = []
    for k in range(K):
        mk = modal == k
        r_k, _ = profile_r(Zv, mk, k)
        k08 = int((post[mk, k] >= 0.8).sum())
        share_k = 100 * float(post[:, k].mean())
        cls_rows.append({'class': CLS[k], 'label': PKG['class_labels'].get(CLS[k], ''), 'children_n': C(sizes[k]),
                         'children_pct': pct_txt(int(sizes[k]), n),
                         'estimated_share_pct': Txt(round(share_k, 1), int(np.floor(share_k + 0.5)) if sizes[k] >= 5 else SUP),
                         'mean_posterior_assigned': num(post[mk, k].mean() if sizes[k] else np.nan, hide=sizes[k] < 5),
                         'posterior_below_0_8_n': C(int(sizes[k]) - k08), 'posterior_ge_0_8_pct': pct_txt(k08, int(sizes[k])),
                         'profile_r_vs_wuhan_same_class': num(r_k if sizes[k] >= MIN_S1 else np.nan, d=3),
                         'wuhan_children_pct': round(100 * WREF['class_n'][CLS[k]] / WREF['children'], 1),
                         'wuhan_picu_children_pct': round(100 * WREF['picu_class_n'][CLS[k]] / WREF['picu_children'], 1)})
    save(cls_rows, '06_classes.csv')
    share_print(cls_rows)

    # Profiles: PIC observed mean standardized value by class (at least 5 children measured in the class) next to the Wuhan means.
    prof_rows = []
    for j, c in enumerate(IND):
        m_j = int(np.isfinite(Zv[:, j]).sum())
        row = {'indicator': LABEL[c], 'pic_measured_pct': pct_txt(m_j, n)}
        for k in range(K):
            mk = (modal == k) & np.isfinite(Zv[:, j])
            row[f'pic_{CLS[k]}'] = num(Zv[mk, j].mean() if mk.sum() else np.nan, hide=mk.sum() < 5)
            row[f'wuhan_{CLS[k]}'] = round(float(MU[k, j]), 3)
        prof_rows.append(row)
    save(prof_rows, '07_profiles.csv')

    # Validators by class and the pre-specified contrasts.
    val_rows, con_rows = [], []
    na_v = 'not estimable (fewer than 5 measured in a group)'
    for v in VALIDATORS:
        lv = P['lv'][v]
        okv = np.isfinite(lv)
        m_all = int(okv.sum())
        row = {'validator': LABEL[v], 'measured_n': C(m_all), 'measured_pct': pct_txt(m_all, n)}
        for k in range(K):
            mk = (modal == k) & okv
            gm = float(np.exp(lv[mk].mean())) if mk.sum() >= 5 else None
            row[f'{CLS[k]}_measured_n'] = C(int(mk.sum()))
            row[f'{CLS[k]}_geometric_mean'] = Txt(round(gm, 3), sig2(gm)) if gm is not None else Txt(None)
        val_rows.append(row)
        for nm, g0 in [('S1 vs S6 (least abnormal)', modal == S6), ('S1 vs all other subphenotypes', modal != S1)]:
            n1_, n0_, e, lo, hi = gm_ratio(lv, modal == S1, g0)
            wk = 'ratio_S1_vs_S6' if 'S6' in nm else 'ratio_S1_vs_others'
            con_rows.append({'validator': LABEL[v], 'contrast': nm, 'S1_measured_n': C(n1_), 'comparison_measured_n': C(n0_),
                             'ratio_of_geometric_means_ci': ci_est(e, lo, hi, na=na_v), 'wuhan_ratio_modal': WREF['validators'][v][wk]})
    save(val_rows, '08_validators_by_class.csv')
    save(con_rows, '09_validator_contrasts.csv')
    share_print(con_rows)

    # In-hospital death by class, S1 vs others and vs S6, and the age-adjusted odds ratio.
    died = COH.DIED.values.astype(bool)
    DM = np.array([[int(died[modal == k].sum()), int((~died[modal == k]).sum())] for k in range(K)])
    death_rows = []
    for k in range(K):
        d_, s_ = int(DM[k, 0]), int(DM[k, 1])
        lo_k, hi_k = cp_ci(d_, int(sizes[k]))
        loc = fmt_ci(100 * d_ / sizes[k], 100 * lo_k, 100 * hi_k, d=1) if sizes[k] else None
        death_rows.append({'row': f'{CLS[k]}', 'children_n': C(sizes[k]), 'deaths_n': C(d_), 'risk_pct': pct_txt(d_, int(sizes[k])),
                           'estimate_ci': Txt(loc, ''),
                           'wuhan_in_hospital_death_pct': WREF['in_hospital_death']['risk_pct_by_class'][CLS[k]],
                           'wuhan_death_or_critical_discharge_pct': WREF['death_or_critical_discharge']['risk_pct_by_class'][CLS[k]]})
    for nm, g0 in [('S1 vs all other subphenotypes', modal != S1), ('S1 vs S6 (least abnormal)', modal == S6)]:
        g1 = modal == S1
        rr, lo, hi = risk_ratio(int(died[g1].sum()), int(g1.sum()), int(died[g0].sum()), int(g0.sum()))
        rr_cell = (ci_local(rr, lo, hi) if lo is not None else
                   Txt('not estimable (no deaths in a group)' if rr is not None else 'not estimable', ''))
        death_rows.append({'row': f'Risk ratio, {nm}', 'children_n': None, 'deaths_n': None, 'risk_pct': None, 'estimate_ci': rr_cell,
                           'wuhan_in_hospital_death_pct': None, 'wuhan_death_or_critical_discharge_pct': None})
    age = COH.AGE_DAYS.values / 365.25
    knots = np.percentile(age, [10, 50, 90])
    Xd = np.column_stack([np.ones(n), (modal == S1).astype(float), age, rcs3(age, knots)])
    b, cov, conv = logistic(died.astype(float), Xd)
    if conv and cov is not None:
        se = float(np.sqrt(cov[1, 1]))
        or_cell = ci_local(float(np.exp(b[1])), float(np.exp(b[1] - 1.96 * se)), float(np.exp(b[1] + 1.96 * se)))
    else:
        t_ = 'not estimable (no convergence; for example, no deaths or only deaths in S1)'
        or_cell = Txt(t_, '')
    death_rows.append({'row': 'Odds ratio S1 vs all other subphenotypes, adjusted for age (restricted cubic spline, 3 knots)',
                       'children_n': None, 'deaths_n': None, 'risk_pct': None, 'estimate_ci': or_cell,
                       'wuhan_in_hospital_death_pct': None, 'wuhan_death_or_critical_discharge_pct': None})
    save(death_rows, '10_death.csv')
    share_print(death_rows)

    # S1 bedside rule against the model's S1 assignment, and death by rule result.
    assess, pos = rule_positive(F0, COH)
    A_n = int(assess.sum())
    s1a, pa = (modal == S1)[assess], pos[assess]
    tp, fn, fp, tn = int((pa & s1a).sum()), int((~pa & s1a).sum()), int((pa & ~s1a).sum()), int((~pa & ~s1a).sum())
    wp = RULE['wuhan_performance']['all']
    rule_rows = [{'measure': 'Children assessable (AST and LDH measured)', 'value_n': C(A_n), 'estimate_ci': None, 'wuhan': wp['assessable']}]
    for nm, x, key in [('Rule positive and S1 (true positive)', tp, 'tp'), ('Rule negative and S1 (false negative)', fn, 'fn'),
                       ('Rule positive, not S1 (false positive)', fp, 'fp'), ('Rule negative, not S1 (true negative)', tn, 'tn')]:
        rule_rows.append({'measure': nm, 'value_n': C(x), 'estimate_ci': None, 'wuhan': wp[key]})
    for nm, k_, n_, key in [('Sensitivity (%)', tp, tp + fn, 'sensitivity'), ('Specificity (%)', tn, tn + fp, 'specificity'),
                            ('Positive predictive value (%)', tp, tp + fp, 'ppv'), ('Negative predictive value (%)', tn, tn + fn, 'npv')]:
        if n_:
            lo_, hi_ = cp_ci(k_, n_)
            cell_ = Txt(fmt_ci(100 * k_ / n_, 100 * lo_, 100 * hi_, d=1), '')
        else:
            cell_ = Txt(None)
        wv = wp[key]
        rule_rows.append({'measure': nm, 'value_n': None, 'estimate_ci': cell_,
                          'wuhan': f'{100 * wv[0]:.1f} ({100 * wv[1]:.1f}-{100 * wv[2]:.1f})'})
    dA = died[assess]
    a_, n1_, c_, n0_ = int(dA[pa].sum()), int(pa.sum()), int(dA[~pa].sum()), int((~pa).sum())
    rr, lo, hi = risk_ratio(a_, n1_, c_, n0_)
    for nm, x in [('Rule positive: children', n1_), ('Rule positive: in-hospital deaths', a_), ('Rule negative: children', n0_),
                  ('Rule negative: in-hospital deaths', c_)]:
        rule_rows.append({'measure': nm, 'value_n': C(x), 'estimate_ci': None, 'wuhan': None})
    wr = RULE['wuhan_performance']['in_hospital_death']['risk_ratio']
    rule_rows.append({'measure': 'In-hospital death, risk ratio rule positive vs negative', 'value_n': None,
                      'estimate_ci': ci_local(rr, lo, hi) if lo is not None else Txt('not estimable', ''),
                      'wuhan': f'{wr[0]:.2f} ({wr[1]:.2f}-{wr[2]:.2f})'})
    rule_rows.append({'measure': 'In-hospital death, rule positive (%)', 'value_n': None, 'estimate_ci': pct_txt(a_, n1_), 'wuhan': None})
    rule_rows.append({'measure': 'In-hospital death, rule negative (%)', 'value_n': None, 'estimate_ci': pct_txt(c_, n0_), 'wuhan': None})
    save(rule_rows, '11_bedside_rule.csv')
    share_print(rule_rows)

    # Model fit in PIC (transport diagnostics).
    llv = P['lse'] / np.maximum(P['nobs'], 1)
    mx = post.max(1)
    fit_rows = [{'measure': 'Log-likelihood per observed value, mean', 'pic': round(float(llv.mean()), 3), 'wuhan': WREF['fit']['loglik_per_value_mean']},
                {'measure': 'Log-likelihood per observed value, median', 'pic': round(float(np.median(llv)), 3), 'wuhan': WREF['fit']['loglik_per_value_median']},
                {'measure': 'Children below the Wuhan 5th percentile of log-likelihood per value (children, rounded)',
                 'pic': C(int((llv < WREF['fit']['loglik_per_value_p05']).sum())), 'wuhan': '5% of children'},
                {'measure': 'Largest posterior probability, mean', 'pic': round(float(mx.mean()), 3), 'wuhan': WREF['fit']['max_posterior_mean']},
                {'measure': 'Largest posterior probability below 0.8 (children, rounded)', 'pic': C(int((mx < 0.8).sum())),
                 'wuhan': f"{100 - WREF['fit']['max_posterior_ge_0_8_pct']:.1f}% of children"},
                {'measure': 'Missing indicator values (%)', 'pic': round(100 * float(np.isnan(Zv).mean()), 1), 'wuhan': WREF['fit']['missing_pct']}]
    save(fit_rows, '12_model_fit.csv')
    Xl = log_scale(P['R'])
    ind_rows = []
    for j, c in enumerate(IND):
        x, zc = Xl[c].values, Zv[:, j]
        okz = np.isfinite(zc)
        m_j = int(okz.sum())
        ind_rows.append({'indicator': LABEL[c], 'measured_n': C(m_j), 'measured_pct': pct_txt(m_j, n),
                         'wuhan_measured_pct': WREF['indicators'].get({'hr_z': 'heart_rate', 'rr_z': 'resp_rate'}.get(c, c), {}).get('pct_measured'),
                         'standardized_mean': num(zc[okz].mean() if m_j else np.nan, hide=m_j < 5),
                         'standardized_sd': num(zc[okz].std(ddof=1) if m_j > 1 else np.nan, hide=m_j < 5),
                         'at_lower_winsor_limit_n': C(int((x[okz] <= PRE[c]['winsor_low']).sum())),
                         'at_upper_winsor_limit_n': C(int((x[okz] >= PRE[c]['winsor_high']).sum()))})
    save(ind_rows, '12b_indicator_scale.csv')

class             label children_n children_pct estimated_share_pct mean_posterior_assigned posterior_below_0_8_n posterior_ge_0_8_pct profile_r_vs_wuhan_same_class wuhan_children_pct wuhan_picu_children_pct
   S1 hyperinflammatory         25                               11                    0.98                    <5                                               0.94                7.6                    11.7
   S2                           65                               28                    0.92                    10                                              0.726               23.7                    32.0
   S3                           35                               15                    0.81                    15                                              0.595               16.1                    11.9
   S4                           70                               28                    0.85                    25                                               0.76    

## Sensitivity analyses (pre-specified)
* **S-a**: children with ADEM or bacterial meningoencephalitis codes left out.
* **S-b**: CK, ammonia, lactate and D-dimer treated as missing (assay and sample-handling differences between the hospitals).
* **S-c**: PIC's own winsorizing limits, means and SDs instead of the Wuhan constants.
* **S-d**: time anchor at the first ICU entry instead of hospital admission (the eligibility rule is applied again).
* **S-e**: children with an encephalopathy code (list A) only (descriptive).
* **S-f**: de novo latent profile analysis in PIC (the fit and choice rules registered for Wuhan). This check does not rely on assignment by the
  Wuhan model: does a class whose profile correlates at least 0.80 with Wuhan S1 emerge?
* **S-g**: hs-CRP results reported as below a limit entered at the PIC limit (no Wuhan reporting floor).

In [12]:
if GATE_OK:
    SENS = [summary_row(PRIM, 'primary')]
    m_a = ~COH.SENS_A_DROP.values
    RESULTS['S-a'] = analyse('S-a: without ADEM or bacterial meningoencephalitis codes', COH[m_a], F0, FC0)
    RESULTS['S-b'] = analyse('S-b: CK, ammonia, lactate and D-dimer treated as missing', COH, F0, FC0, drop=('ck', 'ammonia', 'lactate', 'd_dimer'),
                             compare=True)
    RESULTS['S-c'] = analyse("S-c: PIC's own winsorizing limits, means and SDs", COH, F0, FC0, pre='internal', compare=True)
    RESULTS['S-d'] = (analyse('S-d: time anchor at first ICU entry', COH_ICU, FV['icu_entry'].reindex(COH_ICU.index),
                              FCV['icu_entry'].reindex(COH_ICU.index), compare=True) if len(COH_ICU) >= 2 else None)
    m_e = COH.LIST_A.values
    RESULTS['S-e'] = analyse('S-e: encephalopathy codes only (descriptive)', COH[m_e], F0, FC0) if m_e.sum() >= 2 else None
    RESULTS['S-g'] = analyse('S-g: hs-CRP below a limit entered at the PIC limit (no Wuhan floor)', COH, F0, FC0, floors=False, compare=True)
    kinds = {'S-a': 'subset', 'S-b': 'same', 'S-c': 'same', 'S-d': 'other', 'S-e': 'subset', 'S-g': 'same'}
    for k_ in ['S-a', 'S-b', 'S-c', 'S-d', 'S-e', 'S-g']:
        if RESULTS.get(k_) is not None:
            SENS.append(summary_row(RESULTS[k_], kinds[k_], parent=PRIM))
        else:
            SENS.append({'analysis': k_, 'children_n': None, 'S1_n': None, 'replication_criterion_met': 'not run (fewer than 2 children)'})
    save(SENS, '13_sensitivity.csv')
    share_print(SENS, drop=('il6_ratio_S1_vs_S6', 'il10_ratio_S1_vs_S6'))

                                                           analysis children_n S1_n S1_pct  S1_profile_r_ci replication_criterion_met death_rr_S1_vs_others kappa_vs_primary common_children_n
                                                            Primary        245   25        0.94 (0.90-0.97)                       yes                                    NaN               NaN
           S-a: without ADEM or bacterial meningoencephalitis codes        220   25        0.94 (0.89-0.97)                       yes                                    NaN               NaN
           S-b: CK, ammonia, lactate and D-dimer treated as missing        245   25        0.96 (0.92-0.98)                       yes                                   0.86               245
                   S-c: PIC's own winsorizing limits, means and SDs        245   25        0.94 (0.88-0.97)                       yes                                   0.79               245
                                S-d: time anc

In [13]:
# S-f: de novo latent profile analysis in PIC (the EM of the Wuhan analysis, ae_lpa_tools, which reproduces StepMix 3.0.0
# 'gaussian_diag_nan'): 1 to 6 classes, 100 random starts each (500 when fewer than two starts reach the best log-likelihood), lowest
# BIC among solutions whose smallest class holds at least 5% of the children. Input: the PIC indicators on the Wuhan scale.
def _prep(X):
    X = np.asarray(X, float)
    M = ~np.isnan(X)
    X0 = np.where(M, X, 0.0)
    return X0, M.astype(float), X0 ** 2


def _m_step(data, resp):
    X0, Mf, X02 = data
    rs = resp.T @ Mf
    sx = resp.T @ X0
    means = sx / rs
    covs = ((resp.T @ X02) - 2 * means * sx + means ** 2 * rs) / rs
    return np.clip(resp.sum(axis=0) / resp.shape[0], 1e-15, 1 - 1e-15), means, covs


def _log_emission(X0, Mf, X02, means, covs):
    prec = 1.0 / covs
    quad = X02 @ prec.T - 2 * X0 @ (means * prec).T + Mf @ (means ** 2 * prec).T
    return -0.5 * (Mf @ np.log(2 * np.pi * covs).T + quad)


def _e_step(data, params):
    weights, means, covs = params
    lp = _log_emission(*data, means, covs) + np.log(weights)[None, :]
    lse = logsumexp(lp, axis=1)
    return float(lse.mean()), lp - lse[:, None]


def _em(data, params, abs_tol=1e-10, max_iter=1000):
    lower = -np.inf
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
        for it in range(1, max_iter + 1):
            prev = lower
            lower, log_resp = _e_step(data, params)
            params = _m_step(data, np.exp(log_resp))
            if not np.isfinite(lower) or not np.all(np.isfinite(params[2])) or np.any(params[2] <= 0):
                return np.nan, None, it, np.nan
            if abs(lower - prev) < abs_tol:
                break
        after = _e_step(data, params)[0]
    return lower, params, it, after


def _random_start(data, k, rng):
    n = data[0].shape[0]
    resp = rng.uniform(size=(n, k))
    resp /= resp.sum(axis=1)[:, np.newaxis]
    resp = np.exp(np.log(np.clip(resp, 1e-15, 1 - 1e-15)))
    _, means, covs = _m_step(data, resp)
    return np.ones(k) / k, means, covs


def _fit(X, k, n_init, seed):
    rng = np.random.RandomState(seed)
    data = _prep(X)
    best, best_par, lls = -np.inf, None, []
    for _ in range(n_init):
        low, p, it, after = _em(data, _random_start(data, k, rng))
        lls.append(after)
        if p is not None and (low > best or best_par is None):
            best, best_par = low, p
    return best, best_par, np.array(lls, float)


def de_novo(Zv, seed=SEED):
    """The fit and choice rules registered for Wuhan, on the PIC indicators (Wuhan scale). Indicators without at least two observed values or
    without variation are left out (the mixture cannot be fitted on them). Returns the chosen number of classes (None when no
    solution meets the rules), one row per number of classes, the fits and the indicators used."""
    keep = [j for j in range(Zv.shape[1]) if np.isfinite(Zv[:, j]).sum() >= 2 and np.nanstd(Zv[:, j]) > 0]
    X = Zv[:, keep]
    n, d = X.shape
    rows, fits = [], {}
    for k in range(1, DE_NOVO_K_MAX + 1):
        best, par, lls = _fit(X, k, DE_NOVO_STARTS, seed)
        reached, starts = int((np.abs(lls - best) < 1e-6).sum()), DE_NOVO_STARTS
        if k > 1 and reached < 2:
            best, par, lls = _fit(X, k, DE_NOVO_FALLBACK, seed)
            reached, starts = int((np.abs(lls - best) < 1e-6).sum()), DE_NOVO_FALLBACK
        if par is None:
            rows.append({'classes': k, 'fitted': False})
            continue
        pst, _, _ = posterior(X, *par)
        sizes_ = np.bincount(pst.argmax(1), minlength=k)
        n_par = (k - 1) + 2 * k * d
        bic = -2 * best * n + n_par * np.log(n)
        fits[k] = (par, pst, sizes_)
        smin = int(sizes_.min())
        rows.append({'classes': k, 'fitted': True, 'bic': round(float(bic), 1), 'smallest_class_n': C(smin),
                     'meets_5pct_rule': bool(smin / n >= MIN_CLASS_SHARE), 'starts': starts, 'starts_reaching_best': reached})
    ok = [r for r in rows if r.get('fitted') and r['meets_5pct_rule']]
    k_best = min(ok, key=lambda r: r['bic'])['classes'] if ok else None
    return k_best, rows, fits, keep


if GATE_OK:
    t_ = time.time()
    K_DN, dn_rows, DN_NOTE = None, [], ''
    try:
        K_DN, DN_ROWS, DN_FITS, DN_KEEP = de_novo(PRIM['Z'].values)
        dn_rows = [dict(r, chosen=(r['classes'] == K_DN)) for r in DN_ROWS]
        if len(DN_KEEP) < len(IND):
            DN_NOTE = 'left out (no variation or fewer than 2 values): ' + ', '.join(LABEL[IND[j]] for j in range(len(IND)) if j not in DN_KEEP)
            dn_rows.append({'classes': 'indicators ' + DN_NOTE})
        if K_DN is not None:
            par, pst, sizes_dn = DN_FITS[K_DN]
            r_dn = np.corrcoef(np.vstack([par[1], MU[:, DN_KEEP]]))[:K_DN, K_DN:]         # de novo classes x Wuhan classes
            best_cls = int(np.argmax(r_dn[:, S1]))
            dn_modal = pst.argmax(1)
            s1_all = PRIM['modal'] == S1
            for j in range(K_DN):
                ov = int(((dn_modal == j) & s1_all).sum())
                dn_rows.append({'classes': f'chosen solution, class {j + 1}', 'children_n': C(sizes_dn[j]),
                                'r_with_wuhan_S1': round(float(r_dn[j, S1]), 3), 'closest_wuhan_class': CLS[int(np.argmax(r_dn[j]))],
                                'r_with_closest_wuhan_class': round(float(r_dn[j].max()), 3), 'also_assigned_S1_by_frozen_model_n': C(ov)})
            S1_LIKE = bool(r_dn[best_cls, S1] >= MIN_R)
            dn_rows.append({'classes': 'S1-like class (r >= 0.80 with Wuhan S1) in the chosen solution', 'children_n': None,
                            'r_with_wuhan_S1': round(float(r_dn[best_cls, S1]), 3), 'closest_wuhan_class': 'yes' if S1_LIKE else 'no'})
        else:
            dn_rows.append({'classes': 'no solution met the fit and choice rules'})
    except Exception as e_:                       # the remaining cells (summary, privacy check) must still run
        dn_rows.append({'classes': f'de novo analysis not completed ({type(e_).__name__})'})
    save(dn_rows, '14_de_novo_lpa.csv')
    print(f'De novo LPA: {K_DN if K_DN is not None else "no"} classes chosen; {time.time() - t_:.0f} s')

De novo LPA: 4 classes chosen; 124 s


In [14]:
# Cohort description table and summary.md (the numbers to paste back).
desc_rows = [{'measure': k, 'value': v} for k, v in desc.items()]
save(desc_rows, '03b_cohort.csv')


def t(x, mode='share'):
    x = getattr(x, mode) if isinstance(x, Txt) else x
    return '' if x is None or (not isinstance(x, str) and pd.isna(x)) else str(x)


def md_table(rows, cols, mode='share'):
    if mode == 'share':                          # columns and rows given in the local tables only are left out
        cols = [c for c in cols if any(t(r.get(c), mode) for r in rows) or c in ('description', 'measure', 'row', 'analysis')]
        rows = [r for r in rows if any(t(r.get(c), mode) for c in cols[1:])]
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    lines += ['| ' + ' | '.join(t(r.get(c), mode) for c in cols) + ' |' for r in rows]
    return lines


def summary_lines(mode):
    L_ = []
    if FIXTURE:
        L_ += ['**TEST RUN ON A SYNTHETIC FIXTURE: every number is meaningless and must never be reported.**', '']
    L_ += [f'# PIC_SUB_02 summary ({"shareable: counts rounded to the nearest 5, 1-4 shown as <5; count-based percentages and estimates in the local tables only" if mode == "share" else "LOCAL ONLY, exact: do not share"})', '',
           f'Run {time.strftime("%d %b %Y %H:%M")}, {round((time.time() - T0) / 60, 1)} min; python {platform.python_version()}, '
           f'pandas {pd.__version__}, numpy {np.__version__}.',
           f'Code fingerprint (SHA-256 of the cell sources): `{CODE_SHA256}` ({NB_NAME}). Frozen model: `{PKG_SHA256}`.',
           f'OSF addendum: {OSF_ADDENDUM or "(none: synthetic test)"}', '', '## Registered threshold', '',
           f'Eligible children: **{t(C(N_ELIG), mode)}**{" (rounded to 5)" if mode == "share" else ""}; the registered threshold of {GATE_N} is '
           f'**{"met" if GATE_OK else "not met"}** (judged on the exact count).', '',
           '## Cohort flow', '']
    L_ += md_table(flow_rows, ['step', 'description', 'unit', 'count_n'], mode)
    L_ += ['', '## Eligible children', '']
    L_ += md_table(desc_rows, ['measure', 'value'], mode)
    L_ += ['', '## Code groups', '']
    L_ += md_table(grp_rows, ['code_group', 'children_n', 'children_pct'], mode)
    if not GATE_OK:
        L_ += ['', 'The model was not applied (registered decision P15).']
        return L_
    P_ = RESULTS['primary']
    r_txt = (r_text(P_) if mode == 'local' else r_share(P_)) if (np.isfinite(P_['r']) and (mode == 'local' or P_['n_s1'] >= 5)) else SUP
    L_ += ['', '## Primary outcome: replication of the hyperinflammatory subphenotype (S1)', '',
           f"* Children assigned to S1: **{t(C(SIZES[S1]), mode)}** of {t(C(N_ELIG), mode)}"
           f"{' (' + t(cls_rows[S1]['children_pct'], mode) + '%)' if mode == 'local' else ' (rounded to 5)'}; Wuhan {WREF['S1_pct']}% of all children and "
           f"{WREF['S1_pct_picu']}% of those with a PICU stay.",
           f"* Correlation of their mean profile with the Wuhan S1 profile: **r = {r_txt}** (95% CI: Fisher's z with the bootstrap SD).",
           f"* Pre-specified criterion, judged on exact values: at least {MIN_S1} children in S1: **{'yes' if P_['n_s1'] >= MIN_S1 else 'no'}**; "
           f"r at least {MIN_R:.2f}: **{'yes' if np.isfinite(P_['r']) and P_['r'] >= MIN_R else 'no'}**; replication **{'MET' if P_['replicated'] else 'NOT MET'}**.", '',
           '## Subphenotypes in PIC', '']
    L_ += md_table(cls_rows, ['class', 'label', 'children_n', 'children_pct', 'estimated_share_pct', 'mean_posterior_assigned',
                              'posterior_below_0_8_n', 'profile_r_vs_wuhan_same_class', 'wuhan_children_pct', 'wuhan_picu_children_pct'], mode)
    L_ += ['', '## Validators (ratio of geometric means, 95% bootstrap CI)', '']
    L_ += md_table(con_rows, ['validator', 'contrast', 'S1_measured_n', 'comparison_measured_n', 'ratio_of_geometric_means_ci', 'wuhan_ratio_modal'], mode)
    L_ += ['', '## In-hospital death', '']
    L_ += md_table(death_rows, ['row', 'children_n', 'deaths_n', 'risk_pct', 'estimate_ci', 'wuhan_in_hospital_death_pct',
                                'wuhan_death_or_critical_discharge_pct'], mode)
    L_ += ['', '## S1 bedside rule (at least 2 of AST >= 200 U/L, LDH >= 700 U/L, INR >= 1.5)', '']
    L_ += md_table(rule_rows, ['measure', 'value_n', 'estimate_ci', 'wuhan'], mode)
    L_ += ['', '## Model fit in PIC', '']
    L_ += md_table(fit_rows, ['measure', 'pic', 'wuhan'], mode)
    L_ += ['', '## Sensitivity analyses', '']
    L_ += md_table(SENS, ['analysis', 'children_n', 'S1_n', 'S1_pct', 'S1_profile_r_ci', 'replication_criterion_met', 'il6_ratio_S1_vs_S6',
                          'il10_ratio_S1_vs_S6', 'death_rr_S1_vs_others', 'kappa_vs_primary'], mode)
    L_ += ['', f"## S-f: de novo latent profile analysis ({f'{K_DN} classes chosen' if K_DN is not None else 'no solution chosen'})", '']
    L_ += md_table(dn_rows, ['classes', 'bic', 'smallest_class_n', 'meets_5pct_rule', 'starts_reaching_best', 'chosen', 'children_n',
                             'r_with_wuhan_S1', 'closest_wuhan_class', 'r_with_closest_wuhan_class', 'also_assigned_S1_by_frozen_model_n'], mode)
    L_ += ['', '## Files', '', '* Tables 01-14 (CSV) in this folder; the same tables without suppression are in output_pic_sub02_LOCAL_ONLY '
           '(keep on this computer).',
           '* Shareable form: counts are rounded to the nearest 5 (1-4 shown as <5); percentages and estimates computed from counts (risks, '
           'risk and odds ratios, accuracy of the bedside rule) are in the local tables only; statistics of continuous data (correlations, '
           'ratios of geometric means to 2 significant digits, means, model fit) are shown. Verdicts are judged on exact values.',
           '* Do not share the executed notebook itself (the .ipynb with outputs).']
    return L_


for mode, folder in (('share', OUT), ('local', LOCAL)):
    (folder / 'summary.md').write_text('\n'.join(summary_lines(mode)) + '\n', encoding='utf-8')
info = {'notebook': 'PIC_SUB_02_replication', 'synthetic_fixture': FIXTURE, 'code_sha256': CODE_SHA256, 'notebook_file': NB_NAME,
        'frozen_model_sha256': PKG_SHA256, 'osf_addendum': OSF_ADDENDUM, 'python': platform.python_version(), 'pandas': pd.__version__,
        'numpy': np.__version__, 'minutes': round((time.time() - T0) / 60, 1), 'gate_met': bool(GATE_OK),
        'parameters': {'SEED': SEED, 'N_BOOT': N_BOOT, 'AGE_MIN_DAYS': AGE_MIN_DAYS, 'AGE_MAX_YEARS': AGE_MAX_YEARS, 'LAB_WINDOW_H': LAB_WINDOW_H,
                       'VALIDATOR_WINDOW_H': VALIDATOR_WINDOW_H, 'VITAL_WINDOW_H': VITAL_WINDOW_H, 'MIN_TESTS': MIN_TESTS, 'GATE_N': GATE_N,
                       'MIN_S1': MIN_S1, 'MIN_R': MIN_R}}
for folder in (OUT, LOCAL):
    (folder / 'run_info.json').write_text(json.dumps(info, indent=1, ensure_ascii=False), encoding='utf-8')
print('\n'.join(summary_lines('share')[:30]))

# PIC_SUB_02 summary (shareable: counts rounded to the nearest 5, 1-4 shown as <5; count-based percentages and estimates in the local tables only)

Run 20 Oct 2026 10:48, 2.9 min; python 3.12.10, pandas 2.3.3, numpy 2.4.4.
Code fingerprint (SHA-256 of the cell sources): `6193ed4ce89128519eb66ab93dd5fd1c16044a152e205a8c0217c027d1e42907` (PIC_SUB_02_replication.ipynb). Frozen model: `d02b69ead00732b4b46b8bf67ff46dc46e5727eb43c735247b1d57badbb408f5`.
OSF addendum: https://osf.io/ygtr2

## Registered threshold

Eligible children: **245** (rounded to 5); the registered threshold of 200 is **met** (judged on the exact count).

## Cohort flow

| step | description | unit | count_n |
|---|---|---|---|
| 1 | Admissions with an inclusion code | admissions | 370 |
| 2 | First such admission per child (index admission) | children | 360 |
| 3 | No exclusion code in the index admission | children | 355 |
| 4 | Aged 28 days or older | children | 350 |
| 5 | Younger than 18 years | children | 350 |
| 

In [15]:
# Privacy check of the shareable folder: no identifier columns, no dates; every count is 0, "<5" or a multiple of 5; every
# percentage is a whole number or suppressed.
ID_COL = re.compile(r'SUBJECT|HADM|ICUSTAY|ROW_ID|PATIENT', flags=re.I)
DATE = re.compile(r'\b(19|20|21)\d{2}-\d{2}-\d{2}\b')
COUNT_OK = r'(|0|<5|suppressed|\d*[05])'
PCT_OK = r'(|suppressed|\d+)'
problems, others = [], []
for p in sorted(OUT.iterdir()):
    if not (p.is_file() and _ours.match(p.name)):
        others.append(p.name)
        continue
    text = p.read_text(encoding='utf-8-sig', errors='replace')
    if p.name != 'run_info.json' and DATE.search(text):
        problems.append(f'{p.name}: date-like text')
    if p.suffix == '.csv':
        df = pd.read_csv(p, dtype=str, keep_default_na=False)
        problems += [f'{p.name}: identifier column {c}' for c in df.columns if ID_COL.search(c)]
        for c in df.columns:
            v_ = df[c].str.strip()
            if c.endswith('_n') and not v_.str.fullmatch(COUNT_OK).all():
                problems.append(f'{p.name}: count column {c} holds a value that is not rounded')
            if c.endswith('_pct') and not c.startswith('wuhan') and not v_.str.fullmatch(PCT_OK).all():
                problems.append(f'{p.name}: percentage column {c} holds a value that is not a whole number')
        if {'measure', 'value'} <= set(df.columns):
            for m_, v_ in zip(df.measure, df.value.str.strip()):
                if m_.endswith('_n') and not re.fullmatch(COUNT_OK, v_):
                    problems.append(f'{p.name}: count row {m_} is not rounded')
                if m_.endswith('_pct') and m_ != 'missing_pct_24_indicators' and not re.fullmatch(PCT_OK, v_):
                    problems.append(f'{p.name}: percentage row {m_} is not a whole number')
    if p.name == 'summary.md':
        for m_ in re.finditer(r'\*\*(\d+)\*\*', text):
            if not re.fullmatch(r'\d*[05]', m_.group(1)):
                problems.append(f'summary.md: bold count {m_.group(1)} is not rounded')
if problems:
    raise RuntimeError('Privacy check failed; do not share the outputs:\n' + '\n'.join(problems))
print(f'Privacy check passed for the {len(list(OUT.iterdir())) - len(others)} files written by this notebook in {OUT.name}. '
      f'Send summary.md or those files (never {LOCAL.name}, and not the executed notebook).')
if others:
    print('Other files in the folder were not written by this notebook and were not checked; do not send them: ' + ', '.join(others))

Privacy check passed for the 18 files written by this notebook in output_pic_sub02. Send summary.md or those files (never output_pic_sub02_LOCAL_ONLY, and not the executed notebook).
